# Sigil notebook: 40_notebook_mock
# Mode: mock (offline, no broker)  |  Source: examples/40_notebook_mock.sigil
# Transports: MOCK, MQTT -> overridden to mock (offline)
# 3 broker(s) declared; kernel reaches no network. Use --mode broker to keep MOCK/MQTT.
# Pipelines: windowed  (local :: sensors/temperature -> local :: sensors/aggregated)
#            chained  (pipe windowed -> cache :: analytics/temperatures)
#
# Mock transport notes
# - topics match exactly, so wildcard topics are published literally
# - the bus is in-process; it is cleared at the start and end of a run
# - window results appear only after the outputs cell drains the pipelines
# - use --mode broker to keep the declared transports instead of mock


In [ ]:
%pip install "commlib-py>=0.13.2"


In [ ]:
import json, sys, threading, types, time
import commlib.transports.mock as _mock_transport
from commlib.transports.mock import clear_mock_bus, Publisher, Subscriber, ConnectionParameters

# R17: the stock bus lock is a non-reentrant threading.Lock held while subscriber
# callbacks run, and every pipeline publishes its sink from inside _on_message.
# Patch to an RLock before any endpoint exists; remove when upstream ships one.
if not hasattr(_mock_transport, "_MOCK_BUS_LOCK"):
    raise RuntimeError(
        "commlib.transports.mock._MOCK_BUS_LOCK is missing; the reentrant-lock "
        "workaround requires commlib-py>=0.13.2. Install it and restart the kernel."
    )
_mock_transport._MOCK_BUS_LOCK = threading.RLock()

_NOTEBOOK_MODULES = {}


def _reset_pipelines_package():
    for _name in [n for n in list(sys.modules) if n == "pipelines" or n.startswith("pipelines.")]:
        del sys.modules[_name]
    _pkg = types.ModuleType("pipelines")
    _pkg.__path__ = []
    sys.modules["pipelines"] = _pkg
    _NOTEBOOK_MODULES.clear()


def _load_pipeline(name, source):
    if "pipelines" not in sys.modules:
        _reset_pipelines_package()
    mod = types.ModuleType("pipelines." + name)
    mod.__package__ = "pipelines"
    mod.__file__ = "<notebook:pipelines/%s.py>" % name
    sys.modules["pipelines." + name] = mod
    setattr(sys.modules["pipelines"], name, mod)
    exec(compile(source, mod.__file__, "exec"), mod.__dict__)
    _NOTEBOOK_MODULES[name] = mod
    return mod


def _clear_modules():
    _NOTEBOOK_MODULES.clear()
    for _name in [n for n in list(sys.modules) if n == "pipelines" or n.startswith("pipelines.")]:
        del sys.modules[_name]


In [ ]:
_load_pipeline("windowed", "from datetime import datetime\nimport json\nimport math\nimport re\nimport time\nimport threading\nimport logging\nimport statistics\nimport random\nfrom collections import defaultdict\n\nfrom commlib.transports.mock import (\n    ConnectionParameters as SourceConnParams,\n    Subscriber,\n)\nfrom commlib.transports.mock import (\n    ConnectionParameters as SinkConnParams,\n    Publisher,\n)\n\nlogger = logging.getLogger(\"sigil.pipeline.windowed\")\n\n_MAX_PUBLISH_RETRIES = 1\n_PUBLISH_RETRY_BASE = 0.1\n\n\ndef _get_nested(d, *keys):\n    for k in keys:\n        if isinstance(d, dict):\n            d = d.get(k)\n        else:\n            return None\n    return d\n\n\ndef _coalesce(value, default):\n    return value if value is not None else default\n\n\nclass WindowedPipeline:\n\n    def __init__(self):\n        self.name = \"windowed\"\n        self._running = False\n        self._lock = threading.Lock()\n        self._downstream_listeners = []\n        self._window_data = defaultdict(list)\n        self._window_stop_event = threading.Event()\n        self._window_thread = None\n        self._window_size = 60\n        self._MAX_WINDOW_BUFFER = 10000\n\n        source_conn = SourceConnParams(\n            host=\"localhost\",\n            port=0,\n        )\n\n        self._subscriber = Subscriber(\n            topic=\"sensors/temperature\",\n            on_message=self._on_message,\n            conn_params=source_conn,\n        )\n\n\n        sink_conn = SinkConnParams(\n            host=\"localhost\",\n            port=0,\n        )\n\n        self._publisher = Publisher(\n            topic=\"sensors/aggregated\",\n            conn_params=sink_conn,\n        )\n\n    def start(self):\n        self._running = True\n        self._subscriber.run()\n        self._publisher.run()\n        time.sleep(0.3)\n        self._schedule_window_flush()\n        logger.info(\"Pipeline '%s' started\", self.name)\n\n    def stop(self, timeout=5):\n        self._running = False\n        self._window_stop_event.set()\n        if self._window_thread:\n            self._window_thread.join(timeout=timeout)\n        _endpoints = []\n        _endpoints.append(self._subscriber)\n        _endpoints.append(self._publisher)\n        for ep in _endpoints:\n            t = threading.Thread(target=ep.stop, daemon=True)\n            t.start()\n            t.join(timeout=timeout)\n        logger.info(\"Pipeline '%s' stopped\", self.name)\n\n    def drain(self):\n        self._flush_window()\n        logger.debug(\"Window buffer drained\")\n\n    def _on_message(self, msg):\n        if not isinstance(msg, dict):\n            try:\n                msg = json.loads(msg) if isinstance(msg, (str, bytes)) else msg\n            except (json.JSONDecodeError, TypeError, ValueError) as exc:\n                logger.warning(\"Invalid message (not JSON dict): %s\", exc)\n                return\n            if not isinstance(msg, dict):\n                logger.warning(\"Invalid message type: %s\", type(msg).__name__)\n                return\n\n\n\n\n\n\n\n\n\n\n\n        group_key = \"__all__\"\n        with self._lock:\n            self._window_data[group_key].append(msg)\n            _buf = self._window_data[group_key]\n            if len(_buf) > self._MAX_WINDOW_BUFFER:\n                _dropped = len(_buf) - self._MAX_WINDOW_BUFFER\n                del _buf[:_dropped]\n                logger.warning(\n                    \"Backpressure: dropped %d oldest messages for group '%s'\",\n                    _dropped, group_key,\n                )\n\n    def _schedule_window_flush(self):\n        if self._window_thread and self._window_thread.is_alive():\n            return\n        interval = self._window_size\n        self._window_stop_event.clear()\n        self._window_thread = threading.Thread(\n            target=self._window_flush_loop, args=(interval,), daemon=True\n        )\n        self._window_thread.start()\n\n    def _window_flush_loop(self, interval):\n        while not self._window_stop_event.wait(timeout=interval):\n            if not self._running:\n                break\n            self._flush_window()\n\n    def _flush_window(self):\n        with self._lock:\n            snapshot = dict(self._window_data)\n            self._window_data.clear()\n\n        for group_key, messages in snapshot.items():\n            if not messages:\n                continue\n            self._process_window(group_key, messages)\n\n    def _process_window(self, group_key, messages):\n        try:\n            aggregated = {}\n            _vals_avg_temp = [\n                v for v in (_get_nested(m, 'value') for m in messages)\n                if v is not None and isinstance(v, (int, float))\n            ]\n            aggregated[\"avg_temp\"] = (\n                statistics.mean(_vals_avg_temp)\n                if _vals_avg_temp else 0.0\n            )\n            _vals_max_temp = [\n                v for v in (_get_nested(m, 'value') for m in messages)\n                if v is not None and isinstance(v, (int, float))\n            ]\n            aggregated[\"max_temp\"] = (\n                max(_vals_max_temp)\n                if _vals_max_temp else 0.0\n            )\n            aggregated[\"n\"] = len(messages)\n\n\n\n            result = aggregated\n\n            result[\"_group\"] = group_key\n            result[\"_timestamp\"] = time.time()\n            result[\"_pipeline\"] = self.name\n\n            self._publish_message(result)\n            logger.debug(\"Published aggregate for group '%s'\", group_key)\n\n        except Exception as exc:\n            logger.error(\n                \"DLQ: failed to process message: %s \u2014 %s\", exc, messages,\n            )\n\n\n\n\n\n\n    def _publish_with_retry(self, publisher, payload):\n        \"\"\"Publish with exponential backoff retry.\"\"\"\n        for attempt in range(_MAX_PUBLISH_RETRIES):\n            try:\n                publisher.publish(payload)\n                return\n            except Exception as exc:\n                if attempt < _MAX_PUBLISH_RETRIES - 1:\n                    delay = _PUBLISH_RETRY_BASE * (2 ** attempt) + random.uniform(0, 0.05)\n                    logger.warning(\n                        \"Publish failed (attempt %d/%d): %s \u2014 retrying in %.2fs\",\n                        attempt + 1, _MAX_PUBLISH_RETRIES, exc, delay,\n                    )\n                    time.sleep(delay)\n                else:\n                    logger.error(\n                        \"Publish failed after %d attempts: %s\", _MAX_PUBLISH_RETRIES, exc,\n                    )\n\n    def _notify_downstream(self, msg):\n        for listener in self._downstream_listeners:\n            listener(msg)\n\n\n    def _publish_message(self, msg):\n        self._publish_with_retry(self._publisher, msg)\n        self._notify_downstream(msg)\n")


In [ ]:
_load_pipeline("chained", "from datetime import datetime\nimport json\nimport math\nimport re\nimport time\nimport threading\nimport logging\nimport statistics\nimport random\nfrom collections import defaultdict\n\nfrom commlib.transports.mock import (\n    ConnectionParameters as SinkConnParams,\n    Publisher,\n)\n\nlogger = logging.getLogger(\"sigil.pipeline.chained\")\n\n_MAX_PUBLISH_RETRIES = 7\n_PUBLISH_RETRY_BASE = 0.1\n\n\ndef _get_nested(d, *keys):\n    for k in keys:\n        if isinstance(d, dict):\n            d = d.get(k)\n        else:\n            return None\n    return d\n\n\ndef _coalesce(value, default):\n    return value if value is not None else default\n\n\nclass ChainedPipeline:\n\n    def __init__(self):\n        self.name = \"chained\"\n        self._running = False\n        self._lock = threading.Lock()\n        self._downstream_listeners = []\n\n\n\n        sink_conn = SinkConnParams(\n            host=\"localhost\",\n            port=0,\n        )\n\n        self._publisher = Publisher(\n            topic=\"analytics/temperatures\",\n            conn_params=sink_conn,\n        )\n\n    def start(self):\n        self._running = True\n        self._publisher.run()\n        time.sleep(0.3)\n        logger.info(\"Pipeline '%s' started\", self.name)\n\n    def stop(self, timeout=5):\n        self._running = False\n        _endpoints = []\n        _endpoints.append(self._publisher)\n        for ep in _endpoints:\n            t = threading.Thread(target=ep.stop, daemon=True)\n            t.start()\n            t.join(timeout=timeout)\n        logger.info(\"Pipeline '%s' stopped\", self.name)\n\n    def drain(self):\n        pass\n\n    def _on_message(self, msg):\n        if not isinstance(msg, dict):\n            try:\n                msg = json.loads(msg) if isinstance(msg, (str, bytes)) else msg\n            except (json.JSONDecodeError, TypeError, ValueError) as exc:\n                logger.warning(\"Invalid message (not JSON dict): %s\", exc)\n                return\n            if not isinstance(msg, dict):\n                logger.warning(\"Invalid message type: %s\", type(msg).__name__)\n                return\n\n\n\n\n\n\n\n        if not (_get_nested(msg, 'n') > 1.0):\n            return\n\n\n\n\n        self._process_immediate(msg)\n\n\n\n\n\n    def _process_immediate(self, msg):\n        try:\n            result = msg\n\n            result[\"_timestamp\"] = time.time()\n            result[\"_pipeline\"] = self.name\n\n            self._publish_message(result)\n\n        except Exception as exc:\n            logger.error(\n                \"DLQ: failed to process message: %s \u2014 %s\", exc, msg,\n            )\n\n\n    def _publish_with_retry(self, publisher, payload):\n        \"\"\"Publish with exponential backoff retry.\"\"\"\n        for attempt in range(_MAX_PUBLISH_RETRIES):\n            try:\n                publisher.publish(payload)\n                return\n            except Exception as exc:\n                if attempt < _MAX_PUBLISH_RETRIES - 1:\n                    delay = _PUBLISH_RETRY_BASE * (2 ** attempt) + random.uniform(0, 0.05)\n                    logger.warning(\n                        \"Publish failed (attempt %d/%d): %s \u2014 retrying in %.2fs\",\n                        attempt + 1, _MAX_PUBLISH_RETRIES, exc, delay,\n                    )\n                    time.sleep(delay)\n                else:\n                    logger.error(\n                        \"Publish failed after %d attempts: %s\", _MAX_PUBLISH_RETRIES, exc,\n                    )\n\n    def _notify_downstream(self, msg):\n        for listener in self._downstream_listeners:\n            listener(msg)\n\n\n    def _publish_message(self, msg):\n        self._publish_with_retry(self._publisher, msg)\n        self._notify_downstream(msg)\n")


In [ ]:
from pipelines.windowed import WindowedPipeline
from pipelines.chained import ChainedPipeline

pipeline_instances = {
    "windowed": WindowedPipeline(),
    "chained": ChainedPipeline(),
}
COLLECTED = {}

pipeline_instances["windowed"]._downstream_listeners.append(
    pipeline_instances["chained"]._on_message
)


In [ ]:
# Load-bearing order: clear_mock_bus() -> collectors -> recursion guards ->
# _callback_registered reset -> start() -> inputs. Publishing before the
# subscribers run drops the message, and the preamble's RLock patch is what
# makes a sink publish from inside _on_message safe.
clear_mock_bus()
COLLECTED = {}
_collectors = []


def _collect(topic):
    COLLECTED.setdefault(topic, [])

    def _cb(data):
        COLLECTED[topic].append(data)

    sub = Subscriber(
        topic=topic,
        on_message=_cb,
        conn_params=ConnectionParameters(host="mock", port=0),
    )
    sub.run()
    _collectors.append(sub)


for _t in ["sensors/aggregated", "analytics/temperatures"]:
    _collect(_t)


def _guard_recursion(pipe, limit=32):
    sub = getattr(pipe, "_subscriber", None)
    if sub is None or sub.onmessage is None:
        return
    orig, state = sub.onmessage, {"depth": 0}

    def _wrapped(data):
        state["depth"] += 1
        try:
            if state["depth"] > limit:
                raise RuntimeError("mock self-loop: sink topic feeds its own source")
            return orig(data)
        finally:
            state["depth"] -= 1

    sub.onmessage = _wrapped

_endpoints = list(pipeline_instances.values()) + _collectors
for _ep in _endpoints:
    _sub = getattr(_ep, "_subscriber", None)
    if _sub is not None:
        _sub._callback_registered = False
for _p in pipeline_instances.values():
    _guard_recursion(_p)
    _p.start()


In [ ]:
_pub = Publisher(
    topic="sensors/temperature",
    conn_params=ConnectionParameters(host="mock", port=0),
)
_pub.run()
for _v in (12, 25, 5):
    _pub.publish({"value": _v, "sensor": "windowed", "ts": time.time()})

time.sleep(0.1)


In [ ]:
for _p in pipeline_instances.values():
    _p.drain()
time.sleep(0.1)
try:
    from IPython.display import display
except ImportError:  # plain-python runs (tests, scripts) have no IPython
    display = print

for _t, _msgs in COLLECTED.items():
    print(f"{_t}: {len(_msgs)} message(s)")
    display(_msgs[-5:])


In [ ]:
for _p in pipeline_instances.values():
    try:
        _p.drain()
    except Exception as _exc:
        print("drain failed:", _exc)
    try:
        _p.stop()
    except Exception as _exc:
        print("stop failed:", _exc)
for _s in _collectors:
    _s.stop()
clear_mock_bus()
_clear_modules()
print("mock bus cleared, pipeline modules dropped")
for _t, _msgs in COLLECTED.items():
    print(f"final {_t}: {len(_msgs)} message(s)")  # @finally publishes during stop()
